<a href="https://colab.research.google.com/github/Gauravsingh640/AI-Based-Gym-Workout-Recommendation-System/blob/main/AI_Based_Gym_Workout_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - AI-Based Gym Workout Recommendation System



# **Project Summary -**

Developed an AI-Based Gym Workout Recommendation System that provides personalized exercise recommendations based on a user's fitness goal, target muscle group, available equipment, and difficulty level. The system uses a content-based recommendation approach with TF-IDF Vectorization and Cosine Similarity to suggest the most relevant workouts from a Kaggle fitness dataset, along with exercise instructions and GIFs (where available).

# **GitHub Link -**

Kaggle Link : https://www.kaggle.com/datasets/omarxadel/fitness-exercises-dataset

Github Link : https://github.com/Gauravsingh640/AI-Based-Gym-Workout-Recommendation-System




# **Code Part Begin...**

# **Step 1: Import Libraries**

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import Image, display

# **Step 2: Upload Dataset**

In [3]:
df = pd.read_csv("/content/exercises.csv")

# **Step 3: Explore Dataset**

In [4]:
df.head()

,bodyPart,equipment,gifUrl,id,name,target,secondaryMuscles/0,secondaryMuscles/1,instructions/0,instructions/1,...,instructions/5,secondaryMuscles/2,instructions/6,instructions/7,secondaryMuscles/3,instructions/8,secondaryMuscles/4,instructions/9,secondaryMuscles/5,instructions/10
0,waist,body weight,https://v2.exercisedb.io/image/MOnK4iG0MEt9h8,1,3/4 sit-up,abs,hip flexors,lower back,Lie flat on your back with your knees bent and...,Place your hands behind your head with your el...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,waist,body weight,https://v2.exercisedb.io/image/PERWLDGUxVbpHS,2,45° side bend,abs,obliques,NaN,Stand with your feet shoulder-width apart and ...,Keeping your back straight and your core engag...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,waist,body weight,https://v2.exercisedb.io/image/PLr4yo3j-f1amp,3,air bike,abs,hip flexors,NaN,Lie flat on your back with your hands placed b...,Lift your legs off the ground and bend your kn...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,upper legs,body weight,https://v2.exercisedb.io/image/XPQwM7HECjgNFE,1512,all fours squad stretch,quads,hamstrings,glutes,Start on all fours with your hands directly un...,"Extend one leg straight back, keeping your kne...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,waist,body weight,https://v2.exercisedb.io/image/5nYph4eUGNiEdf,6,alternate heel touchers,abs,obliques,NaN,Lie flat on your back with your knees bent and...,"Extend your arms straight out to the sides, pa...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324 entries, 0 to 1323
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   bodyPart            1324 non-null   object
 1   equipment           1324 non-null   object
 2   gifUrl              1324 non-null   object
 3   id                  1324 non-null   int64 
 4   name                1324 non-null   object
 5   target              1324 non-null   object
 6   secondaryMuscles/0  1324 non-null   object
 7   secondaryMuscles/1  986 non-null    object
 8   instructions/0      1324 non-null   object
 9   instructions/1      1324 non-null   object
 10  instructions/2      1324 non-null   object
 11  instructions/3      1324 non-null   object
 12  instructions/4      1242 non-null   object
 13  instructions/5      739 non-null    object
 14  secondaryMuscles/2  233 non-null    object
 15  instructions/6      313 non-null    object
 16  instructions/7      92 n

In [6]:
df.columns

Index(['bodyPart', 'equipment', 'gifUrl', 'id', 'name', 'target',
       'secondaryMuscles/0', 'secondaryMuscles/1', 'instructions/0',
       'instructions/1', 'instructions/2', 'instructions/3', 'instructions/4',
       'instructions/5', 'secondaryMuscles/2', 'instructions/6',
       'instructions/7', 'secondaryMuscles/3', 'instructions/8',
       'secondaryMuscles/4', 'instructions/9', 'secondaryMuscles/5',
       'instructions/10'],
      dtype='object')

In [7]:
df.shape

(1324, 23)

# **Step 4: Handle Missing Values**

In [8]:
df = df.fillna('')

# **Step 5: Create Combined Features**

In [9]:
df['combined_features'] = (
    df['bodyPart'] + " " +
    df['target'] + " " +
    df['equipment'] + " " +
    df['secondaryMuscles/0'] + " " +
    df['secondaryMuscles/1'] + " " +
    df['secondaryMuscles/2'] + " " +
    df['secondaryMuscles/3'] + " " +
    df['secondaryMuscles/4'] + " " +
    df['secondaryMuscles/5']
)

In [10]:
df[['name', 'combined_features']].head()

,name,combined_features
0,3/4 sit-up,waist abs body weight hip flexors lower back
1,45° side bend,waist abs body weight obliques
2,air bike,waist abs body weight hip flexors
3,all fours squad stretch,upper legs quads body weight hamstrings glutes...
4,alternate heel touchers,waist abs body weight obliques


# **Step 6: TF-IDF Vectorization**

In [11]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

In [12]:
print(tfidf_matrix.shape)

(1324, 93)


# **Step 7: Cosine Similarity**

In [13]:
similarity = cosine_similarity(tfidf_matrix)

In [14]:
print(similarity.shape)

(1324, 1324)


# **Step 8: Recommendation Function**

In [15]:
def recommend_workout(body_part, target, equipment, top_n=5):

    # User input ko ek string me combine karo
    user_input = body_part + " " + target + " " + equipment

    # Dataset + User Input ko combine karo
    all_features = df['combined_features'].tolist()
    all_features.append(user_input)

    # TF-IDF
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(all_features)

    # Similarity
    similarity = cosine_similarity(tfidf_matrix)

    # Last row = User Input
    scores = similarity[-1][:-1]

    # Top Recommendations
    top_indices = scores.argsort()[::-1][:top_n]
    recommendations = df.iloc[top_indices][
        ['name', 'bodyPart', 'target', 'equipment', 'gifUrl']
    ]
    return recommendations

# **Testing**

In [16]:
recommend_workout(
    body_part="waist",
    target="abs",
    equipment="body weight"
)

,name,bodyPart,target,equipment,gifUrl
9,arm slingers hanging bent knee legs,waist,abs,body weight,https://v2.exercisedb.io/image/osybi41Me6RFK2
10,arm slingers hanging straight legs,waist,abs,body weight,https://v2.exercisedb.io/image/yzS2EjySGDLmhh
917,kneeling plank tap shoulder (male),waist,abs,body weight,https://v2.exercisedb.io/image/oUWZjNHiTeHqh6
1130,shoulder tap,waist,abs,body weight,https://v2.exercisedb.io/image/ovo0IVKQmQaACm
1044,pelvic tilt,waist,abs,body weight,https://v2.exercisedb.io/image/KgVMvOCbk8EaWo


# **Step 10: Show GIFs**

In [17]:
results = recommend_workout(
    body_part="waist",
    target="abs",
    equipment="body weight"
)

for _, row in results.iterrows():
    print("=" * 60)
    print("Exercise :", row['name'])
    print("Body Part:", row['bodyPart'])
    print("Target   :", row['target'])
    print("Equipment:", row['equipment'])

    try:
        display(Image(url=row['gifUrl'], width=250))
    except:
        print("GIF not available")

Exercise : arm slingers hanging bent knee legs
Body Part: waist
Target   : abs
Equipment: body weight


Exercise : arm slingers hanging straight legs
Body Part: waist
Target   : abs
Equipment: body weight


Exercise : kneeling plank tap shoulder (male)
Body Part: waist
Target   : abs
Equipment: body weight


Exercise : shoulder tap
Body Part: waist
Target   : abs
Equipment: body weight


Exercise : pelvic tilt
Body Part: waist
Target   : abs
Equipment: body weight


# **Testing**

In [21]:
recommend_workout(
    body_part="chest",
    target="pectorals",
    equipment="barbell"
)

,name,bodyPart,target,equipment,gifUrl
127,barbell guillotine bench press,chest,pectorals,barbell,https://v2.exercisedb.io/image/tZaNqKhurv4Rev
98,barbell bench press,chest,pectorals,barbell,https://v2.exercisedb.io/image/ZZPby9wkLJSFcL
112,barbell decline wide-grip press,chest,pectorals,barbell,https://v2.exercisedb.io/image/-NXtFzKNyzCmBL
108,barbell decline bench press,chest,pectorals,barbell,https://v2.exercisedb.io/image/PNJD51IJJX6j8D
230,barbell wide bench press,chest,pectorals,barbell,https://v2.exercisedb.io/image/YsfYDvRqzjFLGv


In [22]:
sorted(df["bodyPart"].unique())

['back',
 'cardio',
 'chest',
 'lower arms',
 'lower legs',
 'neck',
 'shoulders',
 'upper arms',
 'upper legs',
 'waist']

In [23]:
sorted(df["target"].unique())

['abductors',
 'abs',
 'adductors',
 'biceps',
 'calves',
 'cardiovascular system',
 'delts',
 'forearms',
 'glutes',
 'hamstrings',
 'lats',
 'levator scapulae',
 'pectorals',
 'quads',
 'serratus anterior',
 'spine',
 'traps',
 'triceps',
 'upper back']

In [24]:
sorted(df["equipment"].unique())

['assisted',
 'band',
 'barbell',
 'body weight',
 'bosu ball',
 'cable',
 'dumbbell',
 'elliptical machine',
 'ez barbell',
 'hammer',
 'kettlebell',
 'leverage machine',
 'medicine ball',
 'olympic barbell',
 'resistance band',
 'roller',
 'rope',
 'skierg machine',
 'sled machine',
 'smith machine',
 'stability ball',
 'stationary bike',
 'stepmill machine',
 'tire',
 'trap bar',
 'upper body ergometer',
 'weighted',
 'wheel roller']

In [25]:
recommend_workout(
    body_part="waist",
    target="abs",
    equipment="body weight"
)

,name,bodyPart,target,equipment,gifUrl
9,arm slingers hanging bent knee legs,waist,abs,body weight,https://v2.exercisedb.io/image/osybi41Me6RFK2
10,arm slingers hanging straight legs,waist,abs,body weight,https://v2.exercisedb.io/image/yzS2EjySGDLmhh
917,kneeling plank tap shoulder (male),waist,abs,body weight,https://v2.exercisedb.io/image/oUWZjNHiTeHqh6
1130,shoulder tap,waist,abs,body weight,https://v2.exercisedb.io/image/ovo0IVKQmQaACm
1044,pelvic tilt,waist,abs,body weight,https://v2.exercisedb.io/image/KgVMvOCbk8EaWo
